# Document Classification Comparison

This notebook runs all three classification methods and compares their results.

**Note:** If you encounter errors about missing fields, restart the kernel to ensure the latest code is loaded.

In [1]:
# Install the document_classification module in editable mode
import sys
import subprocess
from pathlib import Path

project_root = Path.cwd().parent.parent.parent

# Install in editable mode
subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", 
    str(project_root / "src" / "document_classification"),
    "-q"
], check=True)

print("✓ Module installed successfully")

✓ Module installed successfully


In [2]:
# Import required libraries
import json
import sys
import time
from pathlib import Path
import pandas as pd
import importlib

# Get the project root directory  
project_root = Path.cwd().parent.parent  # Go up from notebooks -> document_classification -> src
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Working directory: {Path.cwd()}")

# Import document classification components directly
from src.document_classification.utils.config import Config, ClassificationMethod
from src.document_classification.utils.factory import create_classifier
from src.document_classification.acu_classifier import ACUClassifier
from src.document_classification.acu_llm_text_classifier import ACULLMTextClassifier
from src.document_classification.llm_image_classifier import LLMImageClassifier

# Import shared authentication manager
from src.document_classification.utils.auth_manager import SharedAuthManager

print("✓ Imports Successful")

# Initialize shared authentication (this will be reused across all classifiers)
print("Initializing shared authentication...")
shared_auth = SharedAuthManager()
print("✓ Shared authentication manager ready (will avoid multiple auth attempts)")

Project root: /Users/lindamthomas/Documents/GitHub/HSBC_IWPB_UW/NEW/src
Working directory: /Users/lindamthomas/Documents/GitHub/HSBC_IWPB_UW/NEW/src/document_classification/notebooks
✓ Imports Successful
Initializing shared authentication...
✓ Shared authentication manager ready (will avoid multiple auth attempts)


In [3]:
# Configure logging to display in notebook
import logging
import sys

# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Create console handler for notebook output
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

# Create formatter
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(console_handler)

# Also configure specific loggers that might be used in the modules
logging.getLogger('document_classification').setLevel(logging.INFO)
logging.getLogger('src.document_classification').setLevel(logging.INFO)

print("✓ Logging configured for notebook display")

✓ Logging configured for notebook display


## Configuration

## Logging Configuration

The following cell enables detailed logging from the classification modules so you can see what's happening during processing.

In [4]:
# Set up comprehensive logging for all classification modules
import os

# Enable detailed logging for all classification components
loggers_to_configure = [
    'document_classification',
    'src.document_classification', 
    'document_classification.acu_classifier',
    'document_classification.acu_llm_text_classifier',
    'document_classification.llm_image_classifier',
    'document_classification.utils',
    'azure.core',  # For Azure services
]

for logger_name in loggers_to_configure:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)  # Change to DEBUG for even more detail

# Also set environment variable for more verbose output
os.environ['PYTHONUNBUFFERED'] = '1'

print("✓ Comprehensive logging configured")
print("  ℹ️  You should now see detailed logs from classification modules")
print("  ℹ️  Change level to DEBUG above for even more detail")

✓ Comprehensive logging configured
  ℹ️  You should now see detailed logs from classification modules
  ℹ️  Change level to DEBUG above for even more detail


In [5]:
# Test document path
test_doc = "/Users/lindamthomas/Documents/GitHub/HSBC_IWPB_UW/NEW/src/document_classification/data/sample.pdf"

# Verify file exists
if not Path(test_doc).exists():
    print(f"⚠️  Test file not found: {test_doc}")
    print("Please update the path above")
else:
    print(f"✓ Test file: {Path(test_doc).name}")

✓ Test file: test.pdf


## 1. ACU-Only Classification

In [6]:
print("Running ACU-Only classifier...")
print("-" * 60)

try:
    # Enable verbose logging for this classification
    logging.getLogger('document_classification').setLevel(logging.DEBUG)
    logging.getLogger('src.document_classification').setLevel(logging.DEBUG)

    # Set method
    Config.CLASSIFICATION_METHOD = ClassificationMethod.ACU_ONLY

    # Create classifier
    classifier_acu = create_classifier()
    print("✓ ACU Classifier created successfully")

    # Classify the document
    print(f"Classifying document: {Path(test_doc).name}")
    result_acu = classifier_acu.classify({'path': test_doc})

    print(f"✓ Document Type: {result_acu['document_type']}")
    print(f"✓ Confidence: {result_acu['confidence']:.3f}")
    
    # Handle segments if available
    segments = result_acu.get('segments', [])
    print(f"✓ Segments: {len(segments)}")

    # Handle token usage properly
    metadata = result_acu.get('metadata', {})
    token_usage = metadata.get('token_usage', {})
    if token_usage:
        total_tokens = token_usage.get('total_tokens', 0)
        print(f"✓ Tokens: {total_tokens:,}")
    else:
        print("✓ Tokens: N/A")

    # Display timing information if available
    if 'total_duration' in metadata:
        duration = metadata['total_duration']
        acu_time = metadata.get('acu_duration', 0)
        print(f"✓ Total Time: {duration:.2f}s (ACU: {acu_time:.2f}s)")

    # Show segment details if available
    if segments:
        print("\nSegment details:")
        for seg in segments[:5]:  # Show first 5 segments
            pages = f"p{seg['start_page']}" if seg['start_page'] == seg['end_page'] else f"p{seg['start_page']}-{seg['end_page']}"
            print(f"  • {seg['category']} ({pages})")
    
    # Show page classifications if available
    page_classifications = result_acu.get('page_classifications', [])
    if page_classifications:
        print(f"\nPage classifications: {len(page_classifications)} pages")
        for page in page_classifications[:3]:  # Show first 3 pages
            page_num = page.get('page_number', 'N/A')
            doc_type = page.get('document_type', 'N/A')

            print(f"  • Page {page_num}: {doc_type} ")

except Exception as e:
    print(f"✗ ACU Classification failed: {e}")
    import traceback
    print("\nFull error details:")
    print(traceback.format_exc())
    result_acu = {
        'document_type': 'Error',
        'confidence': 0.0,
        'error': str(e)
    }

Running ACU-Only classifier...
------------------------------------------------------------
2025-12-29 00:23:14,269 - src.document_classification.utils.factory - INFO - Creating classifier with method: acu_only
2025-12-29 00:23:14,270 - src.document_classification.utils.auth_manager - INFO - Initializing shared Azure credential...
2025-12-29 00:23:14,270 - azure.identity._credentials.environment - INFO - Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
2025-12-29 00:23:14,288 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2025-12-29 00:23:14,289 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=REDACTED&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.13.11 (macOS-15.7.3-arm64-arm-64bit-Mach-O)'
No body was attached to the reques

## 2. OCR + LLM Text Classification

In [7]:
print("Running ACU + LLM Text classifier...")
print("-" * 60)

try:
    # Set method
    Config.CLASSIFICATION_METHOD = ClassificationMethod.ACU_LLM_TEXT

    # Create classifier
    classifier_text = create_classifier()

    # Classify
    result_text = classifier_text.classify({'path': test_doc})

    print(f"✓ Document Type: {result_text['document_type']}")
    print(f"✓ Confidence: {result_text['confidence']:.3f}")
    
    # Safely access metadata with proper error handling
    metadata = result_text.get('metadata', {})
    successful_pages = metadata.get('successful_pages', metadata.get('pages_processed', 0))
    total_pages = metadata.get('total_pages', metadata.get('total_pages_found', 0))
    print(f"✓ Pages: {successful_pages}/{total_pages}")

    # Updated to access total_tokens from the correct location
    token_usage = metadata.get('token_usage', {})
    total_tokens = token_usage.get('total_tokens', 0)
    print(f"✓ Tokens: {total_tokens:,}")

    # Display timing information if available
    if 'total_duration' in metadata:
        duration = metadata['total_duration']
        ocr_time = metadata.get('total_ocr_time', 0)
        llm_time = metadata.get('total_llm_time', 0)
        print(f"✓ Total Time: {duration:.2f}s (OCR: {ocr_time:.2f}s, LLM: {llm_time:.2f}s)")
    elif 'total_time' in metadata:
        duration = metadata['total_time']
        ocr_time = metadata.get('total_ocr_time', 0)
        llm_time = metadata.get('total_llm_time', 0)
        print(f"✓ Total Time: {duration:.2f}s (OCR: {ocr_time:.2f}s, LLM: {llm_time:.2f}s)")

    print("\nPer-page classifications:")
    page_classifications = result_text.get('page_classifications', [])
    for i, page_class in enumerate(page_classifications[:5]):
        page_num = page_class.get('page', page_class.get('page_number', i+1))
        category = page_class.get('category', page_class.get('document_type', 'N/A'))
        confidence = page_class.get('confidence', 0)
        print(f"  • Page {page_num}: {category} ({confidence:.3f})")

except Exception as e:
    error_msg = str(e).lower()
    import traceback
    print(f"✗ ACU + LLM Text Classification failed: {e}")
    print("\nFull error details:")
    print(traceback.format_exc())

Running ACU + LLM Text classifier...
------------------------------------------------------------
2025-12-29 00:23:55,760 - src.document_classification.utils.factory - INFO - Creating classifier with method: acu_llm_text
2025-12-29 00:23:55,762 - src.document_classification.acu_llm_text_classifier - INFO - ACU+LLM Text Classifier initialized with PDF to image conversion
2025-12-29 00:23:55,764 - src.document_classification.acu_llm_text_classifier - INFO - Starting ACU LLM Text classification for: test.pdf
2025-12-29 00:23:55,765 - src.document_classification.acu_llm_text_classifier - INFO - Step 1-2: Converting PDF to images and extracting text via ACU
2025-12-29 00:23:55,765 - src.document_classification.acu_llm_text_classifier - INFO - Processing PDF to JSON: test.pdf
2025-12-29 00:24:13,475 - src.document_classification.acu_llm_text_classifier - INFO - Converted PDF to 56 images in 17.71s
2025-12-29 00:24:13,552 - src.document_classification.acu_llm_text_classifier - INFO - Extracti

## 3. LLM Image Classification

In [8]:
print("Running LLM Image classifier...")
print("-" * 60)

try:
    # Reload the module to ensure PIL fixes are applied
    import importlib
    from src.document_classification import llm_image_classifier
    importlib.reload(llm_image_classifier)
    print("✓ Reloaded LLM Image classifier with PIL fixes")
    
    # Set up paths
    data_dir = Path("../data")  # Go up from notebooks to document_classification, then to data
    pdf_files = [f for f in data_dir.iterdir() if f.suffix.lower() == '.pdf']
    
    if not pdf_files:
        print(f"No PDF files found in data directory: {data_dir.absolute()}")
        # List what's actually there
        print("Contents:", list(data_dir.iterdir()) if data_dir.exists() else "Directory doesn't exist")
        raise FileNotFoundError("No test PDFs available")
    
    print(f"Found {len(pdf_files)} PDF files: {[f.name for f in pdf_files]}")
    
    # Create config instance and set method
    config = Config()
    config.CLASSIFICATION_METHOD = ClassificationMethod.LLM_IMAGE
    
    # Create classifier with the configured method
    classifier_image = create_classifier(config)

    
    # Classify
    result_image = classifier_image.classify({'path': test_doc})
    
    print(f"✓ Document Type: {result_image['document_type']}")
    print(f"✓ Confidence: {result_image['confidence']:.3f}")
    
    # Safely access metadata
    metadata = result_image.get('metadata', {})
    successful_pages = metadata.get('successful_pages', 0)
    total_pages = metadata.get('total_pages', 0)
    print(f"✓ Pages: {successful_pages}/{total_pages}")
    
    # Token usage
    token_usage = metadata.get('token_usage', {})
    total_tokens = token_usage.get('total_tokens', 0)
    print(f"✓ Tokens: {total_tokens:,}")
    
    # Display timing information if available
    if 'total_duration' in metadata:
        duration = metadata['total_duration']
        conversion_time = metadata.get('conversion_time', 0)
        llm_time = metadata.get('total_llm_time', 0)
        print(f"✓ Total Time: {duration:.2f}s (Conversion: {conversion_time:.2f}s, LLM: {llm_time:.2f}s)")
    
    print("\nPer-page classifications:")
    for page_class in result_image.get('page_classifications', [])[:5]:
        page_num = page_class.get('page', 'N/A')
        category = page_class.get('category', page_class.get('document_type', 'N/A'))
        confidence = page_class.get('confidence', 0)
        print(f"  • Page {page_num}: {category} ({confidence:.3f})")

except Exception as e:
    print(f"✗ LLM Image Classification failed: {e}")
    import traceback
    traceback.print_exc()
    result_image = {
        'document_type': 'Error',
        'confidence': 0.0,
        'error': str(e)
    }

Running LLM Image classifier...
------------------------------------------------------------
✓ Reloaded LLM Image classifier with PIL fixes
Found 2 PDF files: ['test.pdf', 'sample.pdf']
2025-12-29 00:34:26,828 - src.document_classification.utils.factory - INFO - Creating classifier with method: llm_image
2025-12-29 00:34:26,829 - src.document_classification.llm_image_classifier - INFO - ACU+LLM Image Classifier initialized
2025-12-29 00:34:26,829 - src.document_classification.llm_image_classifier - INFO - Classifying document: /Users/lindamthomas/Documents/GitHub/HSBC_IWPB_UW/NEW/src/document_classification/data/test.pdf
2025-12-29 00:34:38,947 - src.document_classification.llm_image_classifier - INFO - Converted PDF to 56 images in 12.12s
2025-12-29 00:34:39,682 - azure.identity._internal.decorators - INFO - AzureCliCredential.get_token succeeded
2025-12-29 00:34:39,683 - azure.identity._credentials.default - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2025-

## Results Comparison

In [9]:
# Load results from output folder
output_dir = Path.cwd().parent / "output"

# Find latest result files for each method
result_files = {
    'acu_only': sorted(output_dir.glob('acu_classification_results_*.json'))[-1] if list(output_dir.glob('acu_classification_results_*.json')) else None,
    'acu_llm_text': sorted(output_dir.glob('acu_llm_text_results_*.json'))[-1] if list(output_dir.glob('acu_llm_text_results_*.json')) else None,
    'llm_image': sorted(output_dir.glob('llm_image_results_*.json'))[-1] if list(output_dir.glob('llm_image_results_*.json')) else None
}

# Load data
results_data = {}
for method, file_path in result_files.items():
    if file_path:
        with open(file_path, 'r') as f:
            results_data[method] = json.load(f)
        print(f"✓ Loaded {method}: {file_path.name}")
    else:
        print(f"⚠️  No results found for {method}")

✓ Loaded acu_only: acu_classification_results_20251229_002355.json
✓ Loaded acu_llm_text: acu_llm_text_results_20251229_003426.json
✓ Loaded llm_image: llm_image_results_20251229_003932.json


## Summary Comparison

Let's compare the results from the different classifiers that completed successfully:

In [10]:
import json
import pandas as pd
from pathlib import Path

# Load the latest results from all three approaches
output_dir = Path("../output")

# Define the latest result files
latest_files = {
    "ACU": "acu_classification_results_20251229_000012.json",
    "LLM Image": "llm_image_results_20251229_000510.json", 
    "ACU+LLM Text": "acu_llm_text_results_20251229_000858.json"
}

# Load all results
all_results = {}
for method, filename in latest_files.items():
    file_path = output_dir / filename
    if file_path.exists():
        with open(file_path, 'r') as f:
            all_results[method] = json.load(f)
        print(f"✅ Loaded {method}: {filename}")
    else:
        print(f"❌ File not found: {filename}")

print(f"\nLoaded results from {len(all_results)} methods")

✅ Loaded ACU: acu_classification_results_20251229_000012.json
✅ Loaded LLM Image: llm_image_results_20251229_000510.json
✅ Loaded ACU+LLM Text: acu_llm_text_results_20251229_000858.json

Loaded results from 3 methods


In [11]:
# Comprehensive Results Analysis and Comparison

def analyze_results(results_data):
    """Analyze results from all classification methods"""
    
    analysis = {}
    
    for method, results in results_data.items():
        print(f"\n{'='*50}")
        print(f"ANALYZING: {method}")
        print(f"{'='*50}")
        
        if not results:
            print("No results data available")
            continue
            
        # Handle both single result and list of results
        if isinstance(results, list):
            method_results = results
        else:
            method_results = [results]
        
        # Initialize metrics
        total_time = 0
        total_tokens = 0
        total_prompt_tokens = 0 
        total_completion_tokens = 0
        documents_processed = len(method_results)
        classification_results = []
        
        for result in method_results:
            # Extract timing information - handle different field names
            metadata = result.get('metadata', {})
            # Try different possible timing field names
            timing = (metadata.get('total_time', 0) or 
                     metadata.get('total_duration', 0) or 
                     result.get('total_duration', 0))
            total_time += timing
            
            # Extract token usage - check both root level and metadata
            token_usage = result.get('token_usage', {}) or metadata.get('token_usage', {})
            if token_usage:
                total_tokens += token_usage.get('total_tokens', 0)
                total_prompt_tokens += token_usage.get('prompt_tokens', 0)
                total_completion_tokens += token_usage.get('completion_tokens', 0)
            
            # Extract classification results
            classification_results.append({
                'document_type': result.get('document_type', 'Unknown'),
                'confidence': result.get('confidence', 0),
                'file_path': metadata.get('file_path', 'Unknown')
            })
        
        # Calculate averages
        avg_time = total_time / documents_processed if documents_processed > 0 else 0
        avg_tokens = total_tokens / documents_processed if documents_processed > 0 else 0
        
        # Store analysis
        analysis[method] = {
            'documents_processed': documents_processed,
            'total_execution_time': round(total_time, 2),
            'avg_execution_time': round(avg_time, 2),
            'total_tokens': total_tokens,
            'total_prompt_tokens': total_prompt_tokens,
            'total_completion_tokens': total_completion_tokens,
            'avg_tokens_per_doc': round(avg_tokens, 2),
            'classifications': classification_results
        }
        
        # Print summary
        print(f"📊 Documents Processed: {documents_processed}")
        print(f"⏱️  Total Execution Time: {round(total_time, 2)}s")
        print(f"⏱️  Average Time per Document: {round(avg_time, 2)}s")
        print(f"🪙 Total Tokens Used: {total_tokens:,}")
        print(f"🪙 Average Tokens per Document: {round(avg_tokens, 2)}")
        if total_prompt_tokens > 0:
            print(f"   - Prompt Tokens: {total_prompt_tokens:,}")
            print(f"   - Completion Tokens: {total_completion_tokens:,}")
        
        # Print classification summary
        print(f"📑 Classification Results:")
        for i, result in enumerate(classification_results):
            doc_name = Path(result['file_path']).name if result['file_path'] != 'Unknown' else f'Document {i+1}'
            print(f"   - {doc_name}: {result['document_type']} (confidence: {result['confidence']:.3f})")
    
    return analysis

# Run the analysis
analysis_results = analyze_results(all_results)


ANALYZING: ACU
📊 Documents Processed: 1
⏱️  Total Execution Time: 24.47s
⏱️  Average Time per Document: 24.47s
🪙 Total Tokens Used: 2,552
🪙 Average Tokens per Document: 2552.0
   - Prompt Tokens: 2,474
   - Completion Tokens: 78
📑 Classification Results:
   - sample.pdf: Application (confidence: 1.000)

ANALYZING: LLM Image
📊 Documents Processed: 1
⏱️  Total Execution Time: 28.43s
⏱️  Average Time per Document: 28.43s
🪙 Total Tokens Used: 9,877
🪙 Average Tokens per Document: 9877.0
   - Prompt Tokens: 9,545
   - Completion Tokens: 332
📑 Classification Results:
   - sample.pdf: Application (confidence: 0.980)

ANALYZING: ACU+LLM Text
📊 Documents Processed: 1
⏱️  Total Execution Time: 58.23s
⏱️  Average Time per Document: 58.23s
🪙 Total Tokens Used: 10,253
🪙 Average Tokens per Document: 10253.0
   - Prompt Tokens: 9,793
   - Completion Tokens: 460
📑 Classification Results:
   - sample.pdf: Application (confidence: 0.588)


In [12]:
# Create Comparative Summary Table

def create_comparison_table(analysis_results):
    """Create a comprehensive comparison table"""
    
    # Prepare data for comparison table
    comparison_data = []
    
    for method, data in analysis_results.items():
        comparison_data.append({
            'Method': method,
            'Documents': data['documents_processed'],
            'Total Time (s)': data['total_execution_time'],
            'Avg Time (s)': data['avg_execution_time'],
            'Total Tokens': f"{data['total_tokens']:,}",
            'Avg Tokens/Doc': data['avg_tokens_per_doc'],
            'Speed Rank': 0,  # Will be calculated
            'Cost Rank': 0    # Will be calculated
        })
    
    # Create DataFrame
    df = pd.DataFrame(comparison_data)
    
    # Calculate rankings (1 = best, lower is better)
    df['Speed Rank'] = df['Total Time (s)'].rank(method='min').astype(int)
    df['Cost Rank'] = df[df['Total Tokens'] != '0']['Total Tokens'].str.replace(',', '').astype(int).rank(method='min').fillna(0).astype(int)
    
    return df

# Create and display comparison table
comparison_df = create_comparison_table(analysis_results)
print("\n" + "="*80)
print("📊 COMPREHENSIVE COMPARISON SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))

# Calculate cost estimates (approximate)
print(f"\n💰 ESTIMATED COST ANALYSIS (Azure OpenAI GPT-4o pricing)")
print("-" * 60)

# Azure OpenAI pricing (approximate): 
# Input: $0.0025 per 1K tokens, Output: $0.01 per 1K tokens
for method, data in analysis_results.items():
    if data['total_tokens'] > 0:
        input_cost = (data['total_prompt_tokens'] / 1000) * 0.0025
        output_cost = (data['total_completion_tokens'] / 1000) * 0.01
        total_cost = input_cost + output_cost
        
        print(f"{method}:")
        print(f"  - Input tokens: {data['total_prompt_tokens']:,} (~${input_cost:.4f})")
        print(f"  - Output tokens: {data['total_completion_tokens']:,} (~${output_cost:.4f})")  
        print(f"  - Total estimated cost: ~${total_cost:.4f}")
        print(f"  - Cost per document: ~${total_cost/data['documents_processed']:.6f}")
        print()
    else:
        print(f"{method}: No token-based costs (uses Azure Content Understanding)")
        print()


📊 COMPREHENSIVE COMPARISON SUMMARY
      Method  Documents  Total Time (s)  Avg Time (s) Total Tokens  Avg Tokens/Doc  Speed Rank  Cost Rank
         ACU          1           24.47         24.47        2,552          2552.0           1          1
   LLM Image          1           28.43         28.43        9,877          9877.0           2          2
ACU+LLM Text          1           58.23         58.23       10,253         10253.0           3          3

💰 ESTIMATED COST ANALYSIS (Azure OpenAI GPT-4o pricing)
------------------------------------------------------------
ACU:
  - Input tokens: 2,474 (~$0.0062)
  - Output tokens: 78 (~$0.0008)
  - Total estimated cost: ~$0.0070
  - Cost per document: ~$0.006965

LLM Image:
  - Input tokens: 9,545 (~$0.0239)
  - Output tokens: 332 (~$0.0033)
  - Total estimated cost: ~$0.0272
  - Cost per document: ~$0.027183

ACU+LLM Text:
  - Input tokens: 9,793 (~$0.0245)
  - Output tokens: 460 (~$0.0046)
  - Total estimated cost: ~$0.0291
  - Cost pe